In [1]:
import os
from importlib import reload
import sys
sys.path.insert(
    0, r'C:\Users\richa\GitHub\py_neuromodulation\pyneuromodulation')
from time import time as time
from importlib import reload as reload

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from pybv import write_brainvision
import mne
import mne_bids

import start_BIDS
import define_M1


def get_all_files(path, suffix, get_bids=False, prefix=None, bids_root=None,
                  verbose=False, extension=None):
    """Return all files in all (sub-)directories of path with given suffixes and prefixes (case-insensitive).

    Args:
        path (string)
        suffix (iterable): e.g. ["vhdr", "edf"] or ".json"
        get_bids (boolean): True if BIDS_Path type should be returned instead of string. Default: False
        bids_root (string/path): Path of BIDS root folder. Only required if get_bids=True.
        prefix (iterable): e.g. ["SelfpacedRota", "ButtonPress] (optional)

    Returns:
        filepaths (list of strings or list of BIDS_Path)
    """

    if isinstance(suffix, str):
        suffix = [suffix]
    if isinstance(prefix, str):
        prefix = [prefix]

    filepaths = []
    for root, dirs, files in os.walk(path):
        for file in files:
            for suff in suffix:
                if file.endswith(suff.lower()):
                    if not prefix:
                        filepaths.append(os.path.join(root, file))
                    else:
                        for pref in prefix:
                            if pref.lower() in file.lower():
                                filepaths.append(os.path.join(root, file))

    bids_paths = filepaths
    if get_bids:
        if not bids_root:
            print(
                "Warning: No root folder given. Please pass bids_root parameter to create a complete BIDS_Path object.")
        bids_paths = []
        for filepath in filepaths:
            entities = mne_bids.get_entities_from_fname(filepath)
            try:
                bids_path = mne_bids.BIDSPath(subject=entities["subject"],
                                              session=entities["session"],
                                              task=entities["task"],
                                              run=entities["run"],
                                              acquisition=entities[
                                                  "acquisition"],
                                              processing=entities[
                                                  "processing"],
                                              suffix=entities["suffix"],
                                              extension=extension,
                                              root=bids_root)
            except ValueError as err:
                print(
                    f"ValueError while creating BIDS_Path object for file {filepath}: {err}")
            else:
                bids_paths.append(bids_path)

    if verbose:
        if not bids_paths:
            print("No corresponding files found.")
        else:
            print('Corresponding files found:')
            for idx, file in enumerate(bids_paths):
                print(idx, ':', os.path.basename(file))

    return bids_paths

In [2]:
out_root = r'C:\Users\richa\OneDrive - Charité - Universitätsmedizin Berlin\Beijing_ECOG_LFP_derivatives\pipeline-MotOnsetPred_2021-04-26'

In [3]:
deriv_root = os.path.join(out_root, 'derivatives')

In [4]:
files = get_all_files(
    path=out_root,
    suffix='vhdr',
    get_bids=True,
    prefix='ButtonPress',
    bids_root=out_root,
    verbose=True,
    extension=None)

Corresponding files found:
0 : sub-FOG006_ses-EphysMedOn_task-ButtonPress_acq-StimOff_run-01_ieeg.vhdr
1 : sub-FOG008_ses-EphysMedOn_task-ButtonPress_acq-StimOff_run-01_ieeg.vhdr
2 : sub-FOG010_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_ieeg.vhdr
3 : sub-FOG013_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_ieeg.vhdr
4 : sub-FOGC001_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_ieeg.vhdr


In [5]:
m1_files = get_all_files(
    path=out_root,
    suffix='m1.tsv',
    get_bids=False,
    prefix=None,
    bids_root=out_root,
    verbose=True,
    extension=None)

Corresponding files found:
0 : sub-FOG006_ses-EphysMedOn_task-ButtonPress_acq-StimOff_run-01_ieeg_m1.tsv
1 : sub-FOG008_ses-EphysMedOn_task-ButtonPress_acq-StimOff_run-01_ieeg_m1.tsv
2 : sub-FOG010_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_ieeg_m1.tsv
3 : sub-FOG013_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_ieeg_m1.tsv
4 : sub-FOGC001_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_ieeg_m1.tsv


# Normalization, no clipping

In [15]:
path_settings = os.path.join(deriv_root, 'feat_no_clip', 'settings.json')

In [16]:
for file in files[2:3]:
    path_df = os.path.join(
        deriv_root, file.update(extension=None).basename+'_m1.tsv')
    start = time()
    start_BIDS.est_features_run(
        file, PATH_M1=path_df, PATH_SETTINGS=path_settings, verbose=True)
    print("Elapsed time: ", time()-start, " seconds")

Reading settings.json.
Testing settings.
No Error occurred when testing the settings.
Extracting parameters from C:\Users\richa\OneDrive - Charité - Universitätsmedizin Berlin\Beijing_ECOG_LFP_derivatives\pipeline-MotOnsetPred_2021-04-26\sub-FOG010\ses-EphysMedOff\ieeg\sub-FOG010_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_ieeg.vhdr...
Setting channel info structure...
Reading events from C:\Users\richa\OneDrive - Charité - Universitätsmedizin Berlin\Beijing_ECOG_LFP_derivatives\pipeline-MotOnsetPred_2021-04-26\sub-FOG010\ses-EphysMedOff\ieeg\sub-FOG010_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_events.tsv.
Reading channel info from C:\Users\richa\OneDrive - Charité - Universitätsmedizin Berlin\Beijing_ECOG_LFP_derivatives\pipeline-MotOnsetPred_2021-04-26\sub-FOG010\ses-EphysMedOff\ieeg\sub-FOG010_ses-EphysMedOff_task-ButtonPress_acq-StimOff_run-01_channels.tsv.
Segment lengths (ms): [1000  500  333  333  333  100  100  100]
No data specified. Sanity checks related to 

# No Normalization

In [6]:
path_settings = os.path.join(deriv_root, 'feat_no_norm', 'settings.json')

In [7]:
for file in files[:]:
    path_df = os.path.join(
        deriv_root, file.update(extension=None).basename+'_m1.tsv')
    start = time()
    start_BIDS.est_features_run(
        file, PATH_M1=path_df, PATH_SETTINGS=path_settings, verbose=True)
    print("Elapsed time: ", time()-start, " seconds")

Reading settings.json.
Testing settings.
No Error occurred when testing the settings.
Extracting parameters from C:\Users\richa\OneDrive - Charité - Universitätsmedizin Berlin\Beijing_ECOG_LFP_derivatives\pipeline-MotOnsetPred_2021-04-26\sub-FOG006\ses-EphysMedOn\ieeg\sub-FOG006_ses-EphysMedOn_task-ButtonPress_acq-StimOff_run-01_ieeg.vhdr...
Setting channel info structure...
Reading events from C:\Users\richa\OneDrive - Charité - Universitätsmedizin Berlin\Beijing_ECOG_LFP_derivatives\pipeline-MotOnsetPred_2021-04-26\sub-FOG006\ses-EphysMedOn\ieeg\sub-FOG006_ses-EphysMedOn_task-ButtonPress_acq-StimOff_run-01_events.tsv.
Reading channel info from C:\Users\richa\OneDrive - Charité - Universitätsmedizin Berlin\Beijing_ECOG_LFP_derivatives\pipeline-MotOnsetPred_2021-04-26\sub-FOG006\ses-EphysMedOn\ieeg\sub-FOG006_ses-EphysMedOn_task-ButtonPress_acq-StimOff_run-01_channels.tsv.
Segment lengths (ms): [1000  500  333  333  333  100  100  100]
No data specified. Sanity checks related to the le